In [ ]:
import torch
from PIL import Image
from msrtv import SE2, SIM2
from msrtv.utils import PSNR
from bm3d import bm3d
from tqdm.auto import tqdm
import tifffile
from pathlib import Path
import json
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "font.size": 10.0,
    "text.latex.preamble": r"\usepackage{lmodern} \usepackage{amssymb} \usepackage{amsmath}",
    "image.cmap": "gray",
    "figure.constrained_layout.use": True,
    "xtick.top": False,
    "xtick.bottom": False,
    "ytick.left": False,
    "ytick.right": False,
    "xtick.labeltop": False,
    "xtick.labelbottom": False,
    "ytick.labelleft": False,
    "ytick.labelright": False,
})
latex_line_width = 4.804

device = "cuda"

In [ ]:
# colours
red = "#B6321C" # BrickRed
blue = "#0071BC" # RoyalBlue
green = "#009B55" # ForestGreen
purple = "#613F99" # RoyalPurple

def plot_image(ax, image, **kwargs):
    image = image.cpu()
    return ax.imshow(image, **kwargs)

output_directory = Path("output")
data_directory = Path("data")

In [ ]:
ground_truth = torch.as_tensor(tifffile.imread(data_directory / "spiral.tif"), dtype=torch.float32, device=device)
noisy = (torch.as_tensor(tifffile.imread(data_directory / "noisyspiral.tif"), dtype=torch.float32, device=device) / 255.).clamp(0., 255.)

In [ ]:
S, Or = 4, 32
H, W = ground_truth.shape
N_spatial = min(H, W)
ξ = 0.51
a = 2.
σ_s = 1.0
σ_o = 0.5
σ_a = 0.5
λ = 50.
cut_freq = 0.5
T = 1.

G_inv_SIM2 = (ξ**-2, ξ**-2, 1., 1e-4)
G_inv_SE2 = (ξ**-2, ξ**-2, 1.)

scales_bm3d = torch.arange(0., 50., 5.)

In [ ]:
cakewavelets = SE2.cakewavelettransform.cakewavelet_stack_fourier(N_spatial, Or)
U_SE2 = SE2.cakewavelettransform.wavelet_transform(noisy, cakewavelets)
simoncelli_filters, lowpass = SIM2.simoncellitransform.steerable_wavelet_stack(H, W, S, Or, cut_freq=0.5, dilation=a)
U_SIM2, U_low = SIM2.simoncellitransform.wavelet_transform(noisy, simoncelli_filters, lowpass)

In [ ]:
ground_truth_SIM2 = SIM2.tvflow.GroundTruth(
    ground_truth=ground_truth,
    u_low=U_low,
    filters=simoncelli_filters,
    lowpass=lowpass,
)

In [ ]:
s = 1
o = 4

fig, ax = plt.subplots(2, 3, figsize=(24, 10))
cbar = plot_image(ax[0, 0], ground_truth)
fig.colorbar(cbar, ax=ax[0, 0])
cbar = plot_image(ax[1, 0], noisy)
fig.colorbar(cbar, ax=ax[1, 0])

cbar = plot_image(ax[0, 1], U_SE2[o])
fig.colorbar(cbar, ax=ax[0, 1])
cbar = plot_image(ax[1, 1], U_SE2.sum(-3) - noisy)
fig.colorbar(cbar, ax=ax[1, 1])

cbar = plot_image(ax[0, 2], U_SIM2[s, o])
fig.colorbar(cbar, ax=ax[0, 2])
cbar = plot_image(ax[1, 2], SIM2.simoncellitransform.inverse_wavelet_transform(
    U_SIM2, U_low, simoncelli_filters, lowpass
) - noisy)
fig.colorbar(cbar, ax=ax[1, 2]);

In [ ]:
def denoise_bm3d(u, ground_truth, scales, max_val=255.):
    PSNRs = []
    for σ in tqdm(scales):
        denoised = bm3d(u, σ)
        PSNRs.append(PSNR(torch.as_tensor(denoised), ground_truth, max_val=max_val))
    return denoised, PSNRs

In [ ]:
U_denoised_SE2, PSNRs_SE2 = SE2.tvflow.denoise(U_SE2, T, G_inv_SE2, σ_s, σ_o, ground_truth)
U_denoised_SIM2, PSNRs_SIM2 = SIM2.tvflow.denoise(U_SIM2, T, G_inv_SIM2, a, σ_s, σ_o, σ_a, ground_truth_SIM2)
denoised_bm3d, PSNRs_bm3d = denoise_bm3d(noisy.cpu(), ground_truth.cpu(), scales_bm3d)

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.plot(PSNRs_SIM2, label="SIM(2)", color=red)
ax.plot(PSNRs_SE2, label="SE(2)", color=green)
ax.plot(PSNRs_bm3d, label="BM3D", color=blue)
ax.legend();